# 02: Fleet exploratory analysis

1. What is the fleet — states, phases, capacity, coverage?
2. Where does it sit relative to the AS/NZS 4777.2 response thresholds?
3. How many intervals survive to the Volt-VAr detection window?
4. **Does the fleet actually do Volt-VAr?** — the question that determines whether
   conformance scoring in D9 is measuring a response or measuring noise.
5. What is the `derating_active` flag, and can it support Method C at D14?

In [ ]:
import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
from solar_edge.lib import se_metadata as meta
from solar_edge.lib import se_queries as q
from solar_edge.lib import se_plots as plots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG
params = se_params.PARAMS
display(se_store.store_status(con)[["logical_name", "exists", "size_mb", "n_rows"]])

## 0. The manifest

Note the `substitution` rows. They record what this delivery does **not** have —
nameplate capacity, a flexible-export flag, irradiance — so their absence is explicit
rather than inferred from a default.

In [ ]:
display(contract.manifest(config, params))

## 1. Build or refresh the site dimension

`s_99` is the 99th percentile of observed apparent power, ported from
`build_s99_estimates.py`. It is an **empirical observation, not a manufacturer
rating** — a site that never reached its inverter limit will have an `s_99` below it,
which biases the apparent-limit test toward finding symptoms. That is why the quantile
is swept in D15.

In [ ]:
REBUILD = False   # set True after rebuilding the store

if REBUILD or not C.store_path("se_site").exists():
    meta.build_site_dimension(con)
    meta.build_capacity_proxies(con)

checks = meta.check_site_dimension(con)
display(checks)
assert checks["pass"].all(), "D4 checks failed."
display(meta.site_summary(con))

## 2. Fleet composition and coverage

In [ ]:
display(q.fleet_composition(con))

coverage = q.monthly_coverage(con)
display(plots.plot_monthly_coverage(coverage))

capacity = q.capacity_distribution(con)
display(plots.plot_capacity_distribution(capacity))

## 3. Timezone resolution check

The fleet diurnal profile in the AEST analysis frame. It must peak at hour 12. A double
hump or a smeared peak would mean the three states were pooled in mismatched frames.

In [ ]:
diurnal = q.diurnal_profile(con)
display(plots.plot_diurnal_profile(diurnal))

## 4. Where the fleet sits relative to the thresholds

This table sizes every downstream claim.

The **240–253 V band** is where Volt-VAr absorption is required and Volt-Watt has not
yet engaged — the window in which Volt-VAr-induced curtailment can be isolated. The
**253–258 V overlap** is where both respond at once; the project notes record that
empirical disaggregation there has no published precedent, which is why the analysis
restricts to 240–253 V.

In [ ]:
bands = q.voltage_band_occupancy(con, config)
display(bands)

vdist = q.voltage_distribution(con, config)
display(plots.plot_voltage_distribution(vdist))

## 5. Cohort funnel

Attrition from the whole store down to the detection window, in the same spirit as
`fetch_population_funnel` in the Solar Analytics work: never present a final count
without showing what it was drawn from.

In [ ]:
display(q.cohort_funnel(con, config, params))

## 6. Does the fleet do Volt-VAr?

The most consequential section in this notebook.

In the CICCADA generator convention a conforming inverter **absorbs** reactive power
(Q < 0) above 240 V, so the median should slope **downward** with rising voltage. This
is also the fleet-scale check on the D2 sign flip.

In [ ]:
signature = q.voltvar_signature(con, config)
display(plots.plot_voltvar_signature(signature))
display(signature)

### 6a. Volt-VAr response, or fixed power factor?

A confound worth ruling out explicitly: site voltage **rises when the site exports
more**, so any reactive power proportional to P traces an apparent slope against
voltage without any Volt-VAr function being active at all.

The discriminator:

- `med_Q_over_P` flat across voltage → fixed power factor, no response
- `med_abs_Q_kvar` rising with voltage → genuine Volt-VAr behaviour

In [ ]:
character = q.reactive_character(con, config)
display(plots.plot_reactive_character(character))
display(character)

print("Median power factor by cohort:")
display(character.groupby("cohort").med_power_factor.median())

### 6b. Single-phase versus three-phase

Flagged during the D2 sign-convention work and still unresolved. The two cohorts move in
**opposite reactive directions**, and three-phase sites are about a quarter of the fleet.

Measured on the full year (see the `reactive_character` table above):

| | single-phase | three-phase |
|---|---|---|
| Median Q at 220–225 V | **−0.166 kvar** (absorbing) | **+0.311 kvar** (supplying) |
| Median Q at 255 V | **−0.216 kvar** | **+0.370 kvar** |
| Median \|Q\| minimum | 0.151 kvar at 230–235 V | 0.275 kvar at 235 V |
| Median power factor | 0.993–0.999 | 0.982–0.998 |

The detail that matters is the third row. **Both cohorts show a minimum in \|Q\| at
230–235 V**, rising on either side — the shape of a Volt-VAr deadband, sitting almost
exactly where AS/NZS 4777.2 puts it (220–240 V). The two curves are close to mirror
images of each other.

That similarity is the point. A cohort responding genuinely *adversely* would not be
expected to reproduce the standard's own deadband geometry; a cohort whose reactive sign
is reported with the opposite polarity would reproduce it exactly, inverted. The evidence
therefore leans toward a **reporting-sign difference in three-phase inverters**, not
toward 405 sites actively doing the wrong thing.

It is not proof, and the consequences of getting it wrong run in opposite directions:

- if the sign is inverted, flipping it makes the cohort broadly conformant, and failing
  to flip would report a spurious fleet-wide non-conformance;
- if the response is genuinely adverse, flipping it would erase a real and significant
  conformance finding.

Aggregate medians cannot settle this. It needs SolarEdge documentation on three-phase
reactive-power sign reporting, or one site with known ground truth.

In [ ]:
display(q.phase_cohort_comparison(con, config))

## 7. The `derating_active` flag

An inverter-reported power-limitation signal with no Solar Analytics counterpart. If it
is voltage-driven it corroborates the Volt-Watt / Volt-VAr story; a flat baseline at
ordinary voltages is something else — thermal, DC clipping, or export limiting — and
must not be attributed to grid response.

Recall from D3 that the raw column is `1.0` or NULL, never `0.0`. "Not derating" and
"not reported" are indistinguishable, so **precision against this flag is interpretable
but recall is not**. Method C at D14 has to state that.

In [ ]:
derating = q.derating_by_voltage(con, config)
display(plots.plot_derating_by_voltage(derating))
display(derating)

In [ ]:
display(q.derating_by_cohort(con, config))

## 8. Sensitivity of the cohort to its own definition

`with_changes()` makes a sweep cheap. Here: how much does the eligible population move
when the cohort definition moves? If a headline number is fragile to these, D15 needs to
say so.

In [ ]:
variants = {
    "default": config,
    "voltage = mean of phases": config.with_changes(voltage_aggregation="mean"),
    "single-phase only": config.with_changes(phase_cohort="single"),
    "three-phase only": config.with_changes(phase_cohort="three"),
    "exclude derating intervals": config.with_changes(derating_selection="exclude"),
    "include night-anomaly sites": config.with_changes(night_anomaly_selection="include"),
    "sites with >= 300 days": config.with_changes(min_days_observed=300),
}

rows = []
for label, variant in variants.items():
    funnel = q.cohort_funnel(con, variant, params)
    band = funnel.iloc[3]
    rows.append({
        "variant": label,
        "cohort_intervals": int(funnel.iloc[1].n_intervals),
        "in_voltvar_window": int(band.n_intervals),
        "sites_in_window": int(band.n_sites),
    })
display(pd.DataFrame(rows))

## 9. Data-quality report (D7)

Every known issue quantified against an explicit threshold, run against the **built
store** so it reports what the analysis will actually see.

Two kinds of row, and the distinction is the point:

- **`ok` / `WARN`** — measurable defects with a threshold. These can pass or fail.
- **`STRUCTURAL`** — properties of the delivery that cannot be fixed and must be carried
  as stated limitations: no nameplate, no irradiance, a derating flag with no explicit
  zero, sparse overnight coverage, timestamps off a common grid.

The structural rows are listed here precisely so that a clean quality report cannot be
mistaken for a complete dataset.

In [ ]:
from solar_edge.lib import se_diagnostics as diag

quality = diag.data_quality_report(con)
display(quality)

n_warn = int((quality.status == "WARN").sum())
n_struct = int((quality.status == "STRUCTURAL").sum())
print(f"{n_warn} warnings, {n_struct} structural limitations")

quality.to_csv(C.ARTEFACT_DIR / "data_quality_report.csv", index=False)
print(f"Written to {C.ARTEFACT_DIR / 'data_quality_report.csv'}")

## What this establishes

- **Population is ample** for the Volt-VAr work: about three quarters of all intervals
  sit in the 240–253 V band, and roughly 1,580 sites survive the cohort filters.
- **The Volt-Watt population is thin** — under 1% of intervals above 253 V, and a few
  thousand above 258 V. D10 conformance rates there will carry wide intervals, and that
  should be stated rather than discovered later.
- **Timezone resolution holds at fleet scale** — the diurnal profile peaks at hour 12.
- **Fleet-median reactive power is small**, with power factor around 0.995–0.997 in both
  cohorts. The strong Volt-VAr responders found during D2 are a minority. Conformance
  rates in D9 must be read against how little most of this fleet is doing.
- **The derating flag is strongly voltage-driven**, which is what makes Method C worth
  building at D14.

## Open question for D9

The single- and three-phase cohorts move in opposite reactive directions. Before the two
are pooled in conformance scoring, this needs resolving — either SolarEdge documentation
on three-phase reactive sign reporting, or a site with known ground truth.

Until then D9 should score the cohorts **separately** and report them separately. Pooling
them would average a real response against its mirror image and understate both.

## Next: D7 — data-quality report

Quantifies the nine issues catalogued in the architecture proposal against explicit
thresholds, including the 20 night-anomaly sites from D3.